In [1]:
from marketsim.simulator.simulator import Simulator

sim = Simulator(
    num_background_agents=50,
    sim_time=1_000,
    num_assets=1,
    lam=0.1,           # arrival intensity for background agents
    mean=100.0,        # long-run fundamental value
    r=0.6,             # mean-reversion strength
    shock_var=10.0,    # volatility of fundamental shocks
    q_max=10,          # max order size for background agents
)

sim.run()

In [33]:
# Inspect market statistics once the run completes
market = sim.markets[0]
first10 = sim.markets[0].get_midprices()[:10]
mid_prices = market.get_midprices()
matched_orders = market.matched_orders

print("First market", market)
print("Total matched orders:", len(matched_orders))
print("First 10 midprices of first market:", first10)  # let's see the difference between this one and the next one
print("Midprices:", mid_prices)


First market <marketsim.market.market.Market object at 0x000002282AFC2B60>
Total matched orders: 136
First 10 midprices of first market: [724.2923821608167, 279.8734922358147, 433.4568734487453, -164.24827336006308, -159.30354879825492, -57.34393461559374, -57.34393461559374, -57.34393461559374, -43.49325898192591, -43.49325898192591]
Midprices: [724.2923821608167, 279.8734922358147, 433.4568734487453, -164.24827336006308, -159.30354879825492, -57.34393461559374, -57.34393461559374, -57.34393461559374, -43.49325898192591, -43.49325898192591, -11.263019524291181, -17.318759967395366, 7.000632995334122, 17.79314588215451, 17.79314588215451, 61.70233536764813, 175.4052028496485, 57.93889742266525, 205.46281325415214, 207.3637555893123, 117.52563824183238, 225.53939893782268, 225.53939893782268, 179.508684743652, 155.76158334776184, -21.657293417088084, 31.806490660085366, 31.806490660085366, -11.45667700229168, -16.91270413317055, 36.51721591468422, 39.179179522033536, 13.036743592259505,

intro_notebook

In [4]:
from marketsim.fourheap.fourheap import FourHeap
from marketsim.fourheap.order import Order
from marketsim.fourheap.constants import BUY, SELL

In [ ]:
# Let's start with order
order1 = Order(price=12, order_type=BUY, quantity=12, time=1, agent_id=1, order_id=1)
order2 = Order(price=32, order_type=SELL, quantity=22, time=1, agent_id=1, order_id=2)
order3 = Order(price=7, order_type=SELL, quantity=7, time=1, agent_id=1, order_id=3)
order4 = Order(price=9, order_type=SELL, quantity=8, time=1, agent_id=1, order_id=4)

In [6]:
# Now let's see initialize the fourheap
fh = FourHeap() 

In [7]:
# Adding an order
fh.insert(order1)
fh.buy_unmatched.order_dict

{1: Order(price=12, order_type=1, quantity=12, agent_id=1, time=1, order_id=1, asset_id=1)}

In [8]:
# Let's add in a sell order
fh.insert(order2)

# Because it's price is higher than the buy price this won't cause a match
fh.sell_unmatched.order_dict

{2: Order(price=32, order_type=-1, quantity=22, agent_id=1, time=1, order_id=2, asset_id=1)}

In [9]:
# Let's add in another sell order
fh.insert(order3)

# Because it's price is lower this will match
print(fh.sell_matched.order_dict)

# Since order 1 is larger some will be matched and some will be unmatched
print(fh.buy_unmatched.order_dict)
print(fh.buy_matched.order_dict)

{3: Order(price=7, order_type=-1, quantity=7, agent_id=1, time=1, order_id=3, asset_id=1)}
{1: Order(price=12, order_type=1, quantity=5, agent_id=1, time=1, order_id=1, asset_id=1)}
{1: Order(price=12, order_type=1, quantity=7, agent_id=1, time=1, order_id=1, asset_id=1)}


In [10]:
# We can remove an order too
fh.remove(3)

# This will unmatch the buy order entirely
print(fh.sell_matched.order_dict)
print(fh.buy_unmatched.order_dict)
print(fh.buy_matched.order_dict)

{}
{1: Order(price=12, order_type=1, quantity=12, agent_id=1, time=1, order_id=1, asset_id=1)}
{}


In [11]:
# We'll add in 3 and 4
fh.insert(order3)
fh.insert(order4)

print(fh.sell_matched.order_dict)
print(fh.sell_unmatched.order_dict)
print(fh.buy_matched.order_dict)

{3: Order(price=7, order_type=-1, quantity=7, agent_id=1, time=1, order_id=3, asset_id=1), 4: Order(price=9, order_type=-1, quantity=5, agent_id=1, time=1, order_id=4, asset_id=1)}
{2: Order(price=32, order_type=-1, quantity=22, agent_id=1, time=1, order_id=2, asset_id=1), 4: Order(price=9, order_type=-1, quantity=3, agent_id=1, time=1, order_id=4, asset_id=1)}
{1: Order(price=12, order_type=1, quantity=12, agent_id=1, time=1, order_id=1, asset_id=1)}


In [12]:
# Now we'll remove order 3
fh.remove(3)

print(fh.sell_matched.order_dict)
print(fh.sell_unmatched.order_dict)
print(fh.buy_matched.order_dict)
print(fh.buy_unmatched.order_dict)

{4: Order(price=9, order_type=-1, quantity=8, agent_id=1, time=1, order_id=4, asset_id=1)}
{2: Order(price=32, order_type=-1, quantity=22, agent_id=1, time=1, order_id=2, asset_id=1)}
{1: Order(price=12, order_type=1, quantity=8, agent_id=1, time=1, order_id=1, asset_id=1)}
{1: Order(price=12, order_type=1, quantity=4, agent_id=1, time=1, order_id=1, asset_id=1)}


In [13]:
# We'll do it in reverse just to see how the sell side works
fh = FourHeap()

order1 = Order(price=12, order_type=SELL, quantity=12, time=1, agent_id=1, order_id=1)
order2 = Order(price=8, order_type=BUY, quantity=22, time=1, agent_id=1, order_id=2)
order3 = Order(price=15, order_type=BUY, quantity=7, time=1, agent_id=1, order_id=3)
order4 = Order(price=24, order_type=BUY, quantity=8, time=1, agent_id=1, order_id=4)

fh.insert(order1)
fh.insert(order2)
fh.insert(order3)
fh.insert(order4)

print(fh.sell_matched.order_dict)
print(fh.sell_unmatched.order_dict)
print(fh.buy_matched.order_dict)
print(fh.buy_unmatched.order_dict)

{1: Order(price=12, order_type=-1, quantity=12, agent_id=1, time=1, order_id=1, asset_id=1)}
{}
{4: Order(price=24, order_type=1, quantity=8, agent_id=1, time=1, order_id=4, asset_id=1), 3: Order(price=15, order_type=1, quantity=4, agent_id=1, time=1, order_id=3, asset_id=1)}
{2: Order(price=8, order_type=1, quantity=22, agent_id=1, time=1, order_id=2, asset_id=1), 3: Order(price=15, order_type=1, quantity=3, agent_id=1, time=1, order_id=3, asset_id=1)}


In [14]:
fh.remove(4)

print(fh.sell_matched.order_dict)
print(fh.sell_unmatched.order_dict)
print(fh.buy_matched.order_dict)
print(fh.buy_unmatched.order_dict)

{1: Order(price=12, order_type=-1, quantity=7, agent_id=1, time=1, order_id=1, asset_id=1)}
{1: Order(price=12, order_type=-1, quantity=5, agent_id=1, time=1, order_id=1, asset_id=1)}
{3: Order(price=15, order_type=1, quantity=7, agent_id=1, time=1, order_id=3, asset_id=1)}
{2: Order(price=8, order_type=1, quantity=22, agent_id=1, time=1, order_id=2, asset_id=1)}


In [15]:
from marketsim.fundamental.lazy_mean_reverting import LazyGaussianMeanReverting

# Let's look at the fundamental next
f = LazyGaussianMeanReverting(final_time=100, mean=12, r=.2, shock_var=.01)

In [16]:
# The fundamental starts at the mean
f.fundamental_values

{0: 12}

In [17]:
# It's only evaluated at times it's called
f.get_value_at(12)
print(f.fundamental_values)

{0: 12, 12: 11.901467862293794}


In [18]:
# When the simulation ends you find the value at the final time step
f.get_final_fundamental()
print(f.fundamental_values)

{0: 12, 12: 11.901467862293794, 100: 11.972585930613024}


test_event_queue

In [19]:
from marketsim.event.event_queue import EventQueue
from marketsim.fourheap.order import Order
from marketsim.fourheap.constants import BUY, SELL
from marketsim.market.market import Market
from marketsim.fundamental.mean_reverting import GaussianMeanReverting

In [ ]:
e = EventQueue()

order1 = Order(price=1, quantity=1, time=1, agent_id=1, order_id=1, order_type=BUY)
order2 = Order(price=1, quantity=1, time=1, agent_id=1, order_id=2, order_type=BUY)
order3 = Order(price=1, quantity=1, time=3, agent_id=1, order_id=3, order_type=SELL)

In [22]:
e.schedule_activity(order1)
e.schedule_activity(order2)
e.schedule_activity(order3)

In [23]:
e.step()

[]

In [24]:
e.step()

[Order(price=1, order_type=1, quantity=1, agent_id=1, time=1, order_id=1, asset_id=1),
 Order(price=1, order_type=1, quantity=1, agent_id=1, time=1, order_id=2, asset_id=1)]

In [25]:
e.step()

[]

In [ ]:
f = GaussianMeanReverting(mean=100, r=0.2, final_time=100, shock_var=10)
m = Market(f, 10)

In [28]:
m.get_fundamental_value()

100.0

In [29]:
m.add_orders([order1, order3, order3])  # the argument should be an iterable: list, tiple, set, generator
#m.add_orders((order1, order3, order3))

In [30]:
m.step()

[]

In [31]:
m.event_queue.scheduled_activities

defaultdict(list,
            {1: [Order(price=1, order_type=1, quantity=1, agent_id=1, time=1, order_id=1, asset_id=1)],
             3: [Order(price=1, order_type=-1, quantity=1, agent_id=1, time=3, order_id=3, asset_id=1),
              Order(price=1, order_type=-1, quantity=1, agent_id=1, time=3, order_id=3, asset_id=1)],
             0: []})

In [32]:
m.get_time()

1

git status
git add .
git commit -m "describe your changes"
git push